In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.stats import boxcox
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import PowerTransformer, StandardScaler

In [5]:
# Ubicación para guardar datos consolidados
ubicacion_datos_consolidados_janis = r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Modelos\modelo_corr_rezagos_relevantes_prom\datos_promedio_rezagos"

In [2]:
df_semanal_meteo_epi = pd.read_excel(
    r"C:\Users\usuario1\OneDrive - Universidad de Antioquia\UNIVERSIDAD DE ANTIOQUIA\Proyecto SAT Dengue\Bases de datos\Datos consolidados\datos_semanal_meteo_epi.xlsx"
)

In [6]:
# ====================== 1. COPIA DE SEGURIDAD ======================
df_procesados = df_semanal_meteo_epi.copy()

# ====================== 2. TRANSFORMACIONES DE NORMALIDAD ======================

# Logaritmo (para variables con muchos ceros)
df_procesados['casos_ln']       = np.log(df_procesados['casos_dengue'] + 1)
df_procesados['prec_ln']        = np.log(df_procesados['prec'] + 1)
df_procesados['dias_lluvia_ln'] = np.log(df_procesados['dias_lluvia'] + 1)

# Box-Cox para variables estrictamente positivas
vars_boxcox = ['temp', 'temp_max', 'temp_min', 'hum_esp', 'hum_rel',
               'vel_vi', 'vel_vi_max', 'vel_vi_min', 'uv']

for var in vars_boxcox:
    transformed, lam = boxcox(df_procesados[var])
    df_procesados[f'{var}_bc'] = transformed
    df_procesados[f'{var}_bc_lambda'] = lam   # Guardamos lambda

# Yeo-Johnson
pt_soi = PowerTransformer(method='yeo-johnson')
pt_sst = PowerTransformer(method='yeo-johnson')

df_procesados['soi_yj'] = pt_soi.fit_transform(df_procesados[['soi']])
df_procesados['sst_yj'] = pt_sst.fit_transform(df_procesados[['sst']])

print("✅ Transformaciones de normalidad aplicadas")

# ====================== 3. ESTANDARIZACIÓN ======================

variables_a_estandarizar = [
    'casos_ln', 'prec_ln', 'dias_lluvia_ln',
    'temp_bc', 'temp_max_bc', 'temp_min_bc', 'hum_esp_bc', 'hum_rel_bc',
    'vel_vi_bc', 'vel_vi_max_bc', 'vel_vi_min_bc', 'uv_bc',
    'soi_yj', 'sst_yj'
]

scaler = StandardScaler()

df_procesados[variables_a_estandarizar] = scaler.fit_transform(
    df_procesados[variables_a_estandarizar]
)

print("✅ Estandarización aplicada correctamente")

# ====================== 4. GUARDAR DOS VERSIONES DEL DATAFRAME ======================

# Asegurarnos que el índice sea de tipo datetime (por si acaso)
if not pd.api.types.is_datetime64_any_dtype(df_procesados.index):
    df_procesados.index = pd.to_datetime(df_procesados.index)

# Crear versión SIN lambda (más limpia para modelado)
df_sin_lambda = df_procesados.drop(
    columns=[col for col in df_procesados.columns if col.endswith('_lambda')]
).copy()

# Guardar versión SIN lambda
df_sin_lambda.to_excel(
    f"{ubicacion_datos_consolidados_janis}/datos_procesados.xlsx", 
    index=True,                    # ← Importante: guardar el índice (fecha)
    sheet_name='datos_procesados'
)

# Guardar versión CON lambda (para invertir después)
df_procesados.to_excel(
    f"{ubicacion_datos_consolidados_janis}/datos_procesados_con_lambda.xlsx", 
    index=True,                    # ← Guardar el índice (fecha)
    sheet_name='datos_procesados'
)

print("✅ Archivos guardados correctamente con fecha como índice:")
print(f"   → datos_procesados.xlsx   (sin columnas _lambda)")
print(f"   → datos_procesados_con_lambda.xlsx    (con lambdas guardados)")



✅ Transformaciones de normalidad aplicadas
✅ Estandarización aplicada correctamente
✅ Archivos guardados correctamente con fecha como índice:
   → datos_procesados.xlsx   (sin columnas _lambda)
   → datos_procesados_con_lambda.xlsx    (con lambdas guardados)


In [7]:
df_modelo = df_procesados.copy()

lags_dict = {
    'hum_esp_bc': range(1, 9),
    'hum_rel_bc': range(1, 9),
    'sst_yj': range(8, 13),
    'vel_vi_max_bc': range(1, 8),
    'vel_vi_bc': range(1, 8),
    'dias_lluvia_ln': range(1, 9),
    'prec_ln': range(4, 9),
    'temp_bc': range(1, 8),
    'temp_max_bc': range(1, 8)
}

for var, lags in lags_dict.items():
    for lag in lags:
        df_modelo[f"{var}_lag_{lag}"] = df_modelo[var].shift(lag)

print("✅ Rezagos creados")

✅ Rezagos creados


In [8]:
df_prom = df_modelo.copy()

df_prom['hum_esp_mean'] = df_prom[[f'hum_esp_bc_lag_{i}' for i in range(1,9)]].mean(axis=1)
df_prom['hum_rel_mean'] = df_prom[[f'hum_rel_bc_lag_{i}' for i in range(1,9)]].mean(axis=1)
df_prom['sst_mean'] = df_prom[[f'sst_yj_lag_{i}' for i in range(8,13)]].mean(axis=1)

df_prom['vel_vi_max_mean'] = df_prom[[f'vel_vi_max_bc_lag_{i}' for i in range(1,8)]].mean(axis=1)
df_prom['vel_vi_mean'] = df_prom[[f'vel_vi_bc_lag_{i}' for i in range(1,8)]].mean(axis=1)

df_prom['dias_lluvia_mean'] = df_prom[[f'dias_lluvia_ln_lag_{i}' for i in range(1,9)]].mean(axis=1)
df_prom['prec_mean'] = df_prom[[f'prec_ln_lag_{i}' for i in range(4,9)]].mean(axis=1)

df_prom['temp_mean'] = df_prom[[f'temp_bc_lag_{i}' for i in range(1,8)]].mean(axis=1)
df_prom['temp_max_mean'] = df_prom[[f'temp_max_bc_lag_{i}' for i in range(1,8)]].mean(axis=1)

print("✅ Promedios creados")

✅ Promedios creados


In [11]:
df_final_prom = df_prom[[
    'casos_ln',
    'hum_esp_mean',
    'hum_rel_mean',
    'sst_mean',
    'vel_vi_max_mean',
    'vel_vi_mean',
    'dias_lluvia_mean',
    'prec_mean',
    'temp_mean',
    'temp_max_mean'
]]

df_final_prom = df_final_prom.dropna()

print("✅ Base final lista")
print(df_final_prom.shape)

✅ Base final lista
(253, 10)


In [20]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = df_final_prom.drop(columns=['casos_ln'])

vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print("\n🔍 VIF de variables:")
print(vif_data.sort_values("VIF", ascending=False))


🔍 VIF de variables:
           Variable         VIF
1      hum_rel_mean  141.309348
8     temp_max_mean  102.617983
7         temp_mean   47.662261
0      hum_esp_mean   41.322260
3   vel_vi_max_mean   17.909772
4       vel_vi_mean   14.994537
5  dias_lluvia_mean    9.374799
6         prec_mean    9.143771
2          sst_mean    3.181135


In [26]:
df_reducido = df_final_prom[[
    'casos_ln',
    'hum_esp_mean',
    'sst_mean',
    'vel_vi_max_mean',
    'dias_lluvia_mean',
    'temp_max_mean'
]]

In [27]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df_reducido.drop(columns=['casos_ln'])

vif_data = pd.DataFrame()
vif_data["Variable"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data.sort_values("VIF", ascending=False))

           Variable       VIF
0      hum_esp_mean  9.367983
4     temp_max_mean  7.880092
3  dias_lluvia_mean  4.960163
2   vel_vi_max_mean  3.791084
1          sst_mean  2.732074
